# Hierarchical Forecasting for Multi-Level Retail Demand

This notebook implements hierarchical forecasting techniques for multi-level retail demand prediction. 

**Key Features:**
- **Approaches:** Implements Bottom-Up, Top-Down, and MinT reconciliation methods.
- **Multi-Level Hierarchy:** Supports aggregation across item, category, store, and region.
- **Temporal Hierarchy:** Aggregates daily forecasts into weekly and monthly views.
- **Cold-Start Handling:** Demonstrates a technique for handling new products.
- **Visualization:** Shows forecast flow-through and propagated uncertainty.

## 1. Setup and Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Specialized libraries
from statsforecast.models import AutoARIMA
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.evaluation import HierarchicalEvaluation
from hierarchicalforecast.utils import aggregate, HierarchicalPlot

import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Data Generation

We generate a synthetic dataset representing daily sales for a retail business with a clear hierarchy: **Region > Store > Category > Item**.

In [ ]:
def generate_hierarchical_data():
    regions = ['North', 'South']
    stores = {'North': ['N1', 'N2'], 'South': ['S1', 'S2']}
    categories = {'N1': ['CatA', 'CatB'], 'N2': ['CatA', 'CatB'], 'S1': ['CatA', 'CatB'], 'S2': ['CatA', 'CatB']}
    items = {'CatA': ['Item1', 'Item2'], 'CatB': ['Item3', 'Item4']}
    
    dates = pd.date_range(start='2021-01-01', periods=365*2, freq='D')
    df_list = []
    
    for region in regions:
        for store in stores[region]:
            for category in categories[store]:
                for item in items[category]:
                    # Create a unique ID for each time series
                    unique_id = f"{region}_{store}_{category}_{item}"
                    
                    # Base sales with seasonality and trend
                    base = 100 + (hash(unique_id) % 50)
                    trend = 0.05 * np.arange(len(dates))
                    seasonality = 20 * np.sin(2 * np.pi * np.arange(len(dates)) / 7) + 10 * np.cos(2 * np.pi * np.arange(len(dates)) / 30.5)
                    noise = np.random.normal(0, 10, len(dates))
                    
                    sales = base + trend + seasonality + noise
                    sales[sales < 10] = 10
                    
                    # Cold-start for a new item
                    if item == 'Item4' and store == 'S2':
                        sales[:365] = 0 # No sales in the first year
                        
                    temp_df = pd.DataFrame({'ds': dates, 'unique_id': unique_id, 'y': sales})
                    df_list.append(temp_df)
                    
    return pd.concat(df_list)

Y_df = generate_hierarchical_data()
print(Y_df.head())

## 3. Defining the Hierarchy

In [ ]:
# Create the specification of the hierarchy
spec = [
    ['region', 'store', 'category', 'item'],
    ['region', 'store', 'category'],
    ['region', 'store'],
    ['region']
]

# Create a DataFrame with the hierarchy structure
S_df = Y_df['unique_id'].drop_duplicates().to_frame()
S_df[['region', 'store', 'category', 'item']] = S_df['unique_id'].str.split('_', expand=True)

# Aggregate the data to get the historical values at all levels
Y_ag_df, S_ag_df, tags = aggregate(Y_df, S_df, spec)

print("Aggregated Hierarchy Spec:")
print(S_ag_df.head())

## 4. Generating Forecasts

We'll split the data into training and testing sets and then generate base forecasts for all levels of the hierarchy.

In [ ]:
Y_train_df = Y_ag_df.groupby('unique_id').apply(lambda x: x.iloc[:-90]).reset_index(drop=True)
Y_test_df = Y_ag_df.groupby('unique_id').apply(lambda x: x.iloc[-90:]).reset_index(drop=True)

# Handle cold start for the new item
cold_start_item = 'South_S2_CatB_Item4'
parent_item = 'South_S2_CatB' # Its parent category

Y_train_cs = Y_train_df.copy()
Y_train_cs.loc[Y_train_cs['unique_id'] == cold_start_item, 'y'] = Y_train_df[Y_train_df['unique_id'] == parent_item]['y'].values / 2

from statsforecast.core import StatsForecast

fcst = StatsForecast(
    df=Y_train_cs,
    models=[AutoARIMA()],
    freq='D',
    n_jobs=-1
)

Y_hat_df = fcst.forecast(90, fitted=True, level=(95,))
Y_hat_df = Y_hat_df.merge(Y_test_df, on=['ds', 'unique_id'], how='left')

## 5. Reconciling Forecasts

In [ ]:
reconcilers = [
    HierarchicalReconciliation(reconciler='bottom_up'),
    HierarchicalReconciliation(reconciler='top_down', top_down_method='forecast_proportions'),
    HierarchicalReconciliation(reconciler='min_trace', min_trace_method='mint_shrink')
]

hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(Y_hat_df=Y_hat_df, Y_df=Y_train_df, S=S_ag_df, tags=tags)

## 6. Evaluation

In [ ]:
evaluator = HierarchicalEvaluation(evaluators=[lambda y, y_hat: np.mean(np.abs(y - y_hat))]) # MAE
evaluation = evaluator.evaluate(Y_hat_df=Y_rec_df, Y_test_df=Y_test_df, tags=tags)
evaluation = evaluation.drop('Overall').reset_index()
evaluation.columns = ['Level', 'AutoARIMA', 'BottomUp', 'TopDown', 'MinT']
print(evaluation)

## 7. Visualization

In [ ]:
hplot = HierarchicalPlot(S=S_ag_df, tags=tags)

# Plot forecasts for a specific store
hplot.plot_series(
    series='South_S1',
    Y_df=Y_rec_df,
    models=['AutoARIMA/min_trace', 'AutoARIMA/bottom_up'],
    level=[95]
)
plt.title('Forecast for Store S1')
plt.show()

# Plot forecasts for a specific category
hplot.plot_series(
    series='North_N2_CatB',
    Y_df=Y_rec_df,
    models=['AutoARIMA/min_trace', 'AutoARIMA/top_down'],
    level=[95]
)
plt.title('Forecast for Category N2/CatB')
plt.show()